# 12 — All-year train-only normalization

Compute DBZ/VEL normalization from the deterministic
internal training split only. Process one annual archive
at a time and preserve validated annual partials in Drive
so interrupted runs can resume.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.7"
VALIDATION_FRACTION = 0.20
VALIDATION_SEED = 20260913
YEARS = tuple(range(2013, 2023))
NUM_WORKERS = 8
FILE_BATCH_SIZE = 8

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        f"tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = (
    BACKUP_ROOT / "manifests"
)
EXPERIMENT_DIRECTORY = (
    BACKUP_ROOT
    / "experiments"
    / "all_year_baseline_v1"
)
PARTIAL_DIRECTORY = (
    EXPERIMENT_DIRECTORY
    / "normalization_partials"
)
NORMALIZATION_PATH = (
    EXPERIMENT_DIRECTORY
    / "normalization.json"
)

if not PACKAGE_PATH.is_file():
    raise FileNotFoundError(
        PACKAGE_PATH
    )

if not MANIFESTS_ROOT.is_dir():
    raise FileNotFoundError(
        MANIFESTS_ROOT
    )

if NORMALIZATION_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite "
        f"{NORMALIZATION_PATH}"
    )

PARTIAL_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

print("package:", PACKAGE_PATH)
print("partials:", PARTIAL_DIRECTORY)
print(
    "normalization:",
    NORMALIZATION_PATH,
)

package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.7-py3-none-any.whl
partials: /content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization_partials
normalization: /content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization.json


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.7-py3-none-any.whl'], returncode=0)

In [4]:
import datetime
import json
import shutil
import tarfile
import time

import numpy as np
import torch
from torch.utils.data import (
    DataLoader,
    Dataset,
)

import tornado_detection
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
    read_netcdf_file,
)

if (
    tornado_detection.__version__
    != PACKAGE_VERSION
):
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

canonical_index = (
    load_canonical_frame_index(
        MANIFESTS_ROOT
    )
)
assigned_index = assign_model_splits(
    canonical_index,
    validation_fraction=(
        VALIDATION_FRACTION
    ),
    seed=VALIDATION_SEED,
)
training_index = assigned_index.loc[
    assigned_index["model_split"].eq(
        "train"
    )
].copy()

assert len(training_index) == 547_672
assert int(
    training_index["frame_label"].sum()
) == 18_230

file_frame_counts = (
    training_index.groupby(
        "archive_member"
    ).size()
)

if not file_frame_counts.eq(4).all():
    raise AssertionError(
        "Internal training files must "
        "contain four frames"
    )

assert (
    training_index[
        "archive_member"
    ].nunique()
    == 136_918
)


class WholeFileDataset(Dataset):
    def __init__(
        self,
        records,
        root,
    ):
        self.records = records
        self.root = root

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        (
            member,
            expected_labels,
        ) = self.records[index]

        result = read_netcdf_file(
            self.root / member
        )

        np.testing.assert_array_equal(
            result.labels,
            expected_labels,
        )

        return torch.from_numpy(
            result.values
        )


print(
    "tornado_detection:",
    tornado_detection.__version__,
)
print(
    "training frames:",
    f"{len(training_index):,}",
)
print(
    "training files:",
    f"{training_index['archive_member'].nunique():,}",
)
print(
    "training positives:",
    f"{int(training_index['frame_label'].sum()):,}",
)

tornado_detection: 0.1.7
training frames: 547,672
training files: 136,918
training positives: 18,230


In [5]:
annual_results = []

for year in YEARS:
    partial_path = (
        PARTIAL_DIRECTORY
        / f"{year}.json"
    )
    year_rows = (
        training_index.loc[
            training_index["year"].eq(
                year
            )
        ]
        .sort_values(
            [
                "archive_member",
                "frame_index",
            ]
        )
    )
    grouped = list(
        year_rows.groupby(
            "archive_member",
            sort=True,
        )
    )

    expected_file_count = len(
        grouped
    )
    expected_frame_count = len(
        year_rows
    )
    expected_positive_count = int(
        year_rows["frame_label"].sum()
    )

    if partial_path.exists():
        partial = json.loads(
            partial_path.read_text()
        )

        expected_provenance = {
            "package_version": (
                PACKAGE_VERSION
            ),
            "year": year,
            "validation_fraction": (
                VALIDATION_FRACTION
            ),
            "validation_seed": (
                VALIDATION_SEED
            ),
            "file_count": (
                expected_file_count
            ),
            "frame_count": (
                expected_frame_count
            ),
            "positive_frame_count": (
                expected_positive_count
            ),
        }

        mismatches = {
            key: {
                "expected": value,
                "actual": partial.get(
                    key
                ),
            }
            for key, value
            in expected_provenance.items()
            if partial.get(key) != value
        }

        if mismatches:
            raise AssertionError(
                f"Invalid existing partial "
                f"{partial_path}: "
                f"{mismatches}"
            )

        annual_results.append(
            partial
        )
        print(
            f"year={year} resumed "
            f"frames="
            f"{expected_frame_count:,}"
        )
        continue

    archive_path = (
        BACKUP_ROOT
        / f"tornet_{year}.tar.gz"
    )
    local_archive = Path(
        f"/content/tornet_{year}.tar.gz"
    )
    extraction_root = Path(
        f"/content/"
        f"tornet_{year}_normalization"
    )

    if not archive_path.is_file():
        raise FileNotFoundError(
            archive_path
        )

    if local_archive.exists():
        local_archive.unlink()

    if extraction_root.exists():
        shutil.rmtree(
            extraction_root
        )

    extraction_root.mkdir(
        parents=True,
        exist_ok=False,
    )

    records = []
    required_members = set()

    for member, rows in grouped:
        labels = (
            rows["frame_label"]
            .astype(np.uint8)
            .to_numpy()
        )

        if labels.shape != (4,):
            raise AssertionError(
                f"Unexpected labels for "
                f"{member}: "
                f"{labels.shape}"
            )

        records.append(
            (member, labels)
        )
        required_members.add(
            member
        )

    copy_started = (
        time.perf_counter()
    )

    shutil.copyfile(
        archive_path,
        local_archive,
    )

    copy_seconds = (
        time.perf_counter()
        - copy_started
    )

    extracted = set()
    extraction_started = (
        time.perf_counter()
    )

    with tarfile.open(
        local_archive,
        mode="r:gz",
    ) as archive:
        for member in archive:
            if (
                not member.isfile()
                or member.name
                not in required_members
            ):
                continue

            destination = (
                extraction_root
                / member.name
            )
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            source_file = (
                archive.extractfile(
                    member
                )
            )

            if source_file is None:
                raise RuntimeError(
                    f"Could not extract "
                    f"{member.name}"
                )

            with (
                source_file,
                destination.open("wb")
                as output_file,
            ):
                shutil.copyfileobj(
                    source_file,
                    output_file,
                    length=1024 * 1024,
                )

            extracted.add(
                member.name
            )

    extraction_seconds = (
        time.perf_counter()
        - extraction_started
    )
    missing = (
        required_members - extracted
    )

    if missing:
        raise RuntimeError(
            "Missing archive members: "
            f"{sorted(missing)[:10]}"
        )

    dataset = WholeFileDataset(
        records,
        extraction_root,
    )
    loader = DataLoader(
        dataset,
        batch_size=FILE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        persistent_workers=True,
    )

    channel_sum = np.zeros(
        4,
        dtype=np.float64,
    )
    channel_sum_squares = np.zeros(
        4,
        dtype=np.float64,
    )
    channel_finite_count = np.zeros(
        4,
        dtype=np.int64,
    )
    processed_frames = 0
    stats_started = (
        time.perf_counter()
    )

    for values in loader:
        finite = torch.isfinite(
            values
        )
        clean = torch.where(
            finite,
            values,
            0.0,
        ).to(torch.float64)

        reduce_dimensions = (
            0,
            1,
            2,
            3,
        )

        channel_sum += (
            clean.sum(
                dim=reduce_dimensions
            ).numpy()
        )
        channel_sum_squares += (
            clean.square()
            .sum(
                dim=reduce_dimensions
            )
            .numpy()
        )
        channel_finite_count += (
            finite.sum(
                dim=reduce_dimensions
            ).numpy()
        )
        processed_frames += int(
            values.shape[0]
            * values.shape[1]
        )

    stats_seconds = (
        time.perf_counter()
        - stats_started
    )

    if (
        processed_frames
        != expected_frame_count
    ):
        raise AssertionError(
            f"Year {year}: "
            f"{processed_frames} != "
            f"{expected_frame_count} "
            f"frames"
        )

    partial = {
        "artifact_kind": (
            "all_year_"
            "normalization_partial"
        ),
        "created_at_utc": (
            datetime.datetime.now(
                datetime.timezone.utc
            ).isoformat()
        ),
        "package_version": (
            PACKAGE_VERSION
        ),
        "year": year,
        "validation_fraction": (
            VALIDATION_FRACTION
        ),
        "validation_seed": (
            VALIDATION_SEED
        ),
        "variables": [
            "DBZ",
            "VEL",
        ],
        "channel_order": [
            "DBZ_sweep_0",
            "DBZ_sweep_1",
            "VEL_sweep_0",
            "VEL_sweep_1",
        ],
        "file_count": (
            expected_file_count
        ),
        "frame_count": (
            expected_frame_count
        ),
        "positive_frame_count": (
            expected_positive_count
        ),
        "channel_sum": (
            channel_sum.tolist()
        ),
        "channel_sum_squares": (
            channel_sum_squares.tolist()
        ),
        "channel_finite_count": (
            channel_finite_count.tolist()
        ),
        "copy_seconds": (
            copy_seconds
        ),
        "extraction_seconds": (
            extraction_seconds
        ),
        "statistics_seconds": (
            stats_seconds
        ),
    }

    partial_path.write_text(
        json.dumps(
            partial,
            indent=2,
            sort_keys=True,
        )
        + "\n"
    )
    annual_results.append(
        partial
    )

    shutil.rmtree(
        extraction_root
    )
    local_archive.unlink()

    print(
        f"year={year} "
        f"files="
        f"{expected_file_count:,} "
        f"frames="
        f"{expected_frame_count:,} "
        f"stats_seconds="
        f"{stats_seconds:.1f}"
    )

year=2013 files=2,764 frames=11,056 stats_seconds=24.4
year=2014 files=13,972 frames=55,888 stats_seconds=104.8
year=2015 files=15,724 frames=62,896 stats_seconds=122.9
year=2016 files=14,703 frames=58,812 stats_seconds=109.1
year=2017 files=13,426 frames=53,704 stats_seconds=102.0
year=2018 files=12,437 frames=49,748 stats_seconds=97.3
year=2019 files=16,564 frames=66,256 stats_seconds=130.3
year=2020 files=14,640 frames=58,560 stats_seconds=119.2
year=2021 files=15,268 frames=61,072 stats_seconds=120.7
year=2022 files=17,420 frames=69,680 stats_seconds=140.0


In [6]:
total_sum = np.sum(
    [
        np.asarray(
            row["channel_sum"],
            dtype=np.float64,
        )
        for row in annual_results
    ],
    axis=0,
)
total_sum_squares = np.sum(
    [
        np.asarray(
            row[
                "channel_sum_squares"
            ],
            dtype=np.float64,
        )
        for row in annual_results
    ],
    axis=0,
)
total_finite_count = np.sum(
    [
        np.asarray(
            row[
                "channel_finite_count"
            ],
            dtype=np.int64,
        )
        for row in annual_results
    ],
    axis=0,
)

total_frames = sum(
    row["frame_count"]
    for row in annual_results
)
total_files = sum(
    row["file_count"]
    for row in annual_results
)
total_positives = sum(
    row["positive_frame_count"]
    for row in annual_results
)

assert total_frames == 547_672
assert total_files == 136_918
assert total_positives == 18_230

means = (
    total_sum
    / total_finite_count
)
variances = (
    total_sum_squares
    / total_finite_count
    - np.square(means)
)
variances = np.maximum(
    variances,
    0.0,
)
standard_deviations = np.sqrt(
    variances
)

pixels_per_channel = (
    total_frames * 120 * 240
)
finite_fractions = (
    total_finite_count
    / pixels_per_channel
)

if not np.isfinite(
    means
).all():
    raise AssertionError(
        "Non-finite normalization mean"
    )

if not np.isfinite(
    standard_deviations
).all():
    raise AssertionError(
        "Non-finite normalization "
        "standard deviation"
    )

if not np.all(
    standard_deviations > 0
):
    raise AssertionError(
        "Non-positive normalization "
        "standard deviation"
    )

normalization = {
    "artifact_kind": (
        "all_year_train_normalization"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": (
        PACKAGE_VERSION
    ),
    "years": list(YEARS),
    "official_source_split": (
        "train"
    ),
    "model_split": "train",
    "validation_fraction": (
        VALIDATION_FRACTION
    ),
    "validation_seed": (
        VALIDATION_SEED
    ),
    "variables": [
        "DBZ",
        "VEL",
    ],
    "channel_order": [
        "DBZ_sweep_0",
        "DBZ_sweep_1",
        "VEL_sweep_0",
        "VEL_sweep_1",
    ],
    "tensor_shape": [
        120,
        240,
        4,
    ],
    "training_frame_count": (
        total_frames
    ),
    "training_file_count": (
        total_files
    ),
    "training_positive_frame_count": (
        total_positives
    ),
    "means": means.tolist(),
    "standard_deviations": (
        standard_deviations.tolist()
    ),
    "finite_fractions": (
        finite_fractions.tolist()
    ),
    "channel_finite_count": (
        total_finite_count.tolist()
    ),
    "annual_partial_files": [
        str(
            PARTIAL_DIRECTORY
            / f"{year}.json"
        )
        for year in YEARS
    ],
}

NORMALIZATION_PATH.write_text(
    json.dumps(
        normalization,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print(
    json.dumps(
        normalization,
        indent=2,
        sort_keys=True,
    )
)
print(
    "wrote:",
    NORMALIZATION_PATH,
)

{
  "annual_partial_files": [
    "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization_partials/2013.json",
    "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization_partials/2014.json",
    "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization_partials/2015.json",
    "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization_partials/2016.json",
    "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization_partials/2017.json",
    "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization_partials/2018.json",
    "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization_partials/2019.json",
    "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization_partials/2020.json",
    "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_baseline_v1/normalization_p